<a href="https://colab.research.google.com/github/tanvi257/Distorted-Visual-Sequence-Pattern-Recognition-using-Deep-Learning/blob/main/cig_aiml_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIG AI/ML Problem Statement — CAPTCHA OCR
## CRNN + CTC  |  Trained from scratch  |  No pretrained weights
**Architecture**: ResNet-style CNN Backbone → BiLSTM → CTC Loss  
**Key improvements**: Residual blocks, CBAM attention, beam-search decode, TTA, cosine-warm restarts

In [1]:
# ── 0. Install / unzip ────────────────────────────────────────────────────────
!pip install -q editdistance
!unzip -q cig_ps.zip -d .  2>/dev/null || true

In [2]:
import os, glob, random, string, time
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFilter, ImageEnhance
import editdistance

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
CUDA available: True


In [11]:
!unzip cig_ps.zip

Streaming output truncated to the last 5000 lines.
  inflating: cig_ps/train_images/train-5499.png  
  inflating: cig_ps/train_images/train-55.png  
  inflating: cig_ps/train_images/train-550.png  
  inflating: cig_ps/train_images/train-5500.png  
  inflating: cig_ps/train_images/train-5501.png  
  inflating: cig_ps/train_images/train-5502.png  
  inflating: cig_ps/train_images/train-5503.png  
  inflating: cig_ps/train_images/train-5504.png  
  inflating: cig_ps/train_images/train-5505.png  
  inflating: cig_ps/train_images/train-5506.png  
  inflating: cig_ps/train_images/train-5507.png  
  inflating: cig_ps/train_images/train-5508.png  
  inflating: cig_ps/train_images/train-5509.png  
  inflating: cig_ps/train_images/train-551.png  
  inflating: cig_ps/train_images/train-5510.png  
  inflating: cig_ps/train_images/train-5511.png  
  inflating: cig_ps/train_images/train-5512.png  
  inflating: cig_ps/train_images/train-5513.png  
  inflating: cig_ps/train_images/train-5514.png  
  i

In [12]:
# ── 1. CONFIG ─────────────────────────────────────────────────────────────────
TRAIN_IMG_DIR   = "/content/cig_ps/train_images"
TRAIN_LABEL_CSV = "/content/cig_ps/train-labels.csv"
TEST_IMG_DIR    = "/content/cig_ps/test_images"
YOUR_NAME       = "YourName"        # ← FILL IN
YOUR_ENROLL     = "YourEnrollNo"    # ← FILL IN

IMG_H, IMG_W = 64, 256

CHARS       = string.ascii_uppercase + string.digits   # A-Z 0-9  (36 chars)
BLANK_IDX   = 0
CHAR2IDX    = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR    = {i+1: c for i, c in enumerate(CHARS)}
NUM_CLASSES = len(CHARS) + 1   # 37

BATCH_SIZE   = 32
NUM_EPOCHS   = 80
LR           = 3e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT    = 0.1
PATIENCE     = 15

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [13]:
# ── 2. Label helpers ──────────────────────────────────────────────────────────
def encode_label(text):
    return [CHAR2IDX[c] for c in text.upper() if c in CHAR2IDX]

def decode_ctc_greedy(indices):
    """Standard CTC greedy decoder (collapse blanks & repeats)."""
    out, prev = [], None
    for idx in indices:
        if idx != prev:
            if idx != BLANK_IDX:
                out.append(IDX2CHAR.get(idx, "?"))
            prev = idx
    return "".join(out)

def greedy_decode_batch(log_probs):
    """log_probs: (T, B, C) → list[str] length B"""
    preds = log_probs.argmax(2).permute(1, 0)   # (B, T)
    return [decode_ctc_greedy(p.tolist()) for p in preds]

def compute_cer(preds, targets):
    """Character Error Rate using editdistance."""
    total_dist = sum(editdistance.eval(p, t) for p, t in zip(preds, targets))
    total_len  = max(sum(len(t) for t in targets), 1)
    return total_dist / total_len

In [14]:
# ── 3. Dataset & augmentation ─────────────────────────────────────────────────
class CaptchaDataset(Dataset):
    """
    labels: list of (filename, text_str)  or  None (test mode)
    """
    def __init__(self, img_dir, labels=None, augment=False):
        self.img_dir = img_dir
        self.labels  = labels
        self.augment = augment
        if labels is None:
            self.files = sorted(
                glob.glob(os.path.join(img_dir, "*.png")) +
                glob.glob(os.path.join(img_dir, "*.jpg"))
            )
        else:
            self.files = [os.path.join(img_dir, f) for f, _ in labels]

        self.base_tf = T.Compose([
            T.Grayscale(1),
            T.Resize((IMG_H, IMG_W)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.files)

    def _augment_pil(self, img: Image.Image) -> Image.Image:
        """PIL-level augmentation applied BEFORE resize."""
        # Random rotation ±5°
        if random.random() < 0.4:
            angle = random.uniform(-5, 5)
            img = img.rotate(angle, fillcolor=255)
        # Brightness / contrast jitter
        if random.random() < 0.4:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
        if random.random() < 0.4:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
        return img

    def _augment_np(self, img_np: np.ndarray) -> np.ndarray:
        """Numpy-level augmentation after resize to (IMG_H, IMG_W)."""
        # Gaussian blur
        if random.random() < 0.3:
            k = random.choice([3, 5])
            img_np = cv2.GaussianBlur(img_np, (k, k), 0)
        # Additive noise
        if random.random() < 0.4:
            noise = np.random.randint(-20, 20, img_np.shape, dtype=np.int16)
            img_np = np.clip(img_np.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        # Morphological ops
        if random.random() < 0.25:
            kernel = np.ones((2, 2), np.uint8)
            op = random.choice([cv2.erode, cv2.dilate])
            img_np = op(img_np, kernel, iterations=1)
        # Horizontal shift
        if random.random() < 0.3:
            shift = int(IMG_W * 0.05 * random.uniform(-1, 1))
            M = np.float32([[1, 0, shift], [0, 1, 0]])
            img_np = cv2.warpAffine(img_np, M, (IMG_W, IMG_H),
                                    borderMode=cv2.BORDER_REPLICATE)
        # Random cutout column (simulates ink dropout)
        if random.random() < 0.2:
            col_start = random.randint(0, IMG_W - 20)
            col_w     = random.randint(5, 20)
            img_np[:, col_start:col_start+col_w] = 200
        return img_np

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("L")

        if self.augment:
            img = self._augment_pil(img)
            img_np = self._augment_np(
                np.array(img.resize((IMG_W, IMG_H), Image.BILINEAR))
            )
            img = Image.fromarray(img_np)

        img_t = self.base_tf(img)

        if self.labels is None:
            return img_t, os.path.basename(self.files[idx])

        _, label_str = self.labels[idx]
        label_str = label_str.upper()
        return img_t, torch.tensor(encode_label(label_str), dtype=torch.long), label_str


def collate_train(batch):
    imgs, labels, strs = zip(*batch)
    return (
        torch.stack(imgs),
        torch.cat(labels),
        torch.tensor([len(l) for l in labels], dtype=torch.long),
        list(strs),
    )

def collate_test(batch):
    imgs, fnames = zip(*batch)
    return torch.stack(imgs), list(fnames)

In [15]:
# ── 4. Model: ResNet-CRNN ─────────────────────────────────────────────────────

class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class ResBlock(nn.Module):
    """Basic residual block with optional channel projection."""
    def __init__(self, channels, dropout=0.1):
        super().__init__()
        self.conv1 = ConvBNReLU(channels, channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.drop  = nn.Dropout2d(dropout)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.drop(self.conv2(self.conv1(x))))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, r=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // r, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        w = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * w


class ResNetCRNN(nn.Module):
    """
    CNN backbone: stem + 4 stages of residual blocks + SE attention.
    Sequence model: 2-layer bidirectional LSTM.
    Output: CTC log-probs (T, B, NUM_CLASSES).
    All weights randomly initialised — no pretrained backbone.
    """
    def __init__(self, num_classes, rnn_hidden=256, rnn_layers=2, dropout=0.3):
        super().__init__()

        # ── CNN Backbone ──────────────────────────────────────────────────────
        self.stem = nn.Sequential(
            ConvBNReLU(1, 32, k=3, s=1, p=1),
            ConvBNReLU(32, 64, k=3, s=1, p=1),
            nn.MaxPool2d(2, 2),   # 64×32×128
        )

        self.stage1 = nn.Sequential(
            ConvBNReLU(64, 128),
            ResBlock(128, dropout=0.05),
            SEBlock(128),
            nn.MaxPool2d(2, 2),   # 128×16×64
        )

        self.stage2 = nn.Sequential(
            ConvBNReLU(128, 256),
            ResBlock(256, dropout=0.1),
            ResBlock(256, dropout=0.1),
            SEBlock(256),
            nn.MaxPool2d(2, 2),   # 256×8×32
        )

        self.stage3 = nn.Sequential(
            ConvBNReLU(256, 512),
            ResBlock(512, dropout=0.1),
            ResBlock(512, dropout=0.1),
            SEBlock(512),
            nn.MaxPool2d((2, 1), (2, 1)),  # 512×4×32
        )

        self.stage4 = nn.Sequential(
            ConvBNReLU(512, 512),
            ResBlock(512, dropout=0.1),
            SEBlock(512),
            nn.MaxPool2d((2, 1), (2, 1)),  # 512×2×32
            ConvBNReLU(512, 512),
            nn.AdaptiveAvgPool2d((1, None)),  # 512×1×T
        )

        # ── Sequence model ────────────────────────────────────────────────────
        self.rnn = nn.LSTM(
            512, rnn_hidden, rnn_layers,
            batch_first=False, bidirectional=True,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(rnn_hidden * 2, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, p in m.named_parameters():
                    if   "weight_ih" in name: nn.init.xavier_uniform_(p.data)
                    elif "weight_hh" in name: nn.init.orthogonal_(p.data)
                    elif "bias"      in name: nn.init.zeros_(p.data)

    def forward(self, x):
        # x: (B, 1, H, W)
        f = self.stem(x)
        f = self.stage1(f)
        f = self.stage2(f)
        f = self.stage3(f)
        f = self.stage4(f)          # (B, 512, 1, T)
        f = f.squeeze(2)            # (B, 512, T)
        f = f.permute(2, 0, 1)      # (T, B, 512)
        out, _ = self.rnn(f)        # (T, B, 2*H)
        out = self.fc(self.drop(out))  # (T, B, C)
        return F.log_softmax(out, dim=2)


# Quick shape test
with torch.no_grad():
    _m = ResNetCRNN(NUM_CLASSES)
    _x = torch.randn(2, 1, IMG_H, IMG_W)
    _y = _m(_x)
    print(f"Model output shape: {_y.shape}  (T={_y.shape[0]}, B=2, C={_y.shape[2]})")
    total_params = sum(p.numel() for p in _m.parameters())
    print(f"Total parameters: {total_params:,}")
del _m, _x, _y

Model output shape: torch.Size([32, 2, 37])  (T=32, B=2, C=37)
Total parameters: 26,356,997


In [16]:
# ── 5. Loss & training utilities ──────────────────────────────────────────────

ctc_loss_fn = nn.CTCLoss(blank=BLANK_IDX, reduction="mean", zero_infinity=True)


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_cer = n = 0
    for imgs, lbl, llen, strs in loader:
        imgs = imgs.to(DEVICE)
        lbl  = lbl.to(DEVICE)
        llen = llen.to(DEVICE)

        log_probs = model(imgs)          # (T, B, C)
        B  = imgs.size(0)
        T  = log_probs.size(0)
        input_len = torch.full((B,), T, dtype=torch.long, device=DEVICE)

        loss = ctc_loss_fn(log_probs, lbl, input_len, llen)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        with torch.no_grad():
            preds = greedy_decode_batch(log_probs.detach().cpu())
            total_loss += loss.item() * B
            total_cer  += compute_cer(preds, strs) * B
            n += B

    return total_loss / n, total_cer / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total_cer = n = 0
    for imgs, lbl, llen, strs in loader:
        imgs = imgs.to(DEVICE)
        lbl  = lbl.to(DEVICE)
        llen = llen.to(DEVICE)

        log_probs = model(imgs)
        B  = imgs.size(0)
        T  = log_probs.size(0)
        input_len = torch.full((B,), T, dtype=torch.long, device=DEVICE)

        total_loss += ctc_loss_fn(log_probs, lbl, input_len, llen).item() * B
        total_cer  += compute_cer(greedy_decode_batch(log_probs.cpu()), strs) * B
        n += B

    return total_loss / n, total_cer / n

In [17]:
# ── 6. Test-Time Augmentation (TTA) ───────────────────────────────────────────

@torch.no_grad()
def predict_with_tta(model, imgs_cpu, n_aug=4):
    """
    Average log-probs over n_aug slightly-perturbed versions of each image.
    imgs_cpu: (B, 1, H, W) tensor, already normalised.
    Returns list[str] of length B.
    """
    model.eval()
    B = imgs_cpu.size(0)
    sum_probs = None

    # Original
    lp = model(imgs_cpu.to(DEVICE)).cpu()  # (T, B, C)
    sum_probs = torch.exp(lp)

    # Augmented versions
    aug_tf = T.Compose([
        T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.5),
        T.RandomAffine(degrees=3, translate=(0.03, 0.0), fill=-1.0),
    ])
    for _ in range(n_aug - 1):
        aug_imgs = torch.stack([aug_tf(imgs_cpu[i]) for i in range(B)])
        lp_aug   = model(aug_imgs.to(DEVICE)).cpu()
        sum_probs = sum_probs + torch.exp(lp_aug)

    avg_log_probs = torch.log(sum_probs / n_aug + 1e-9)
    return greedy_decode_batch(avg_log_probs)

In [18]:
# ── 7. Main training loop ─────────────────────────────────────────────────────

def main(predict_only=False):
    model = ResNetCRNN(NUM_CLASSES).to(DEVICE)

    if predict_only:
        model.load_state_dict(torch.load("best_crnn_model.pth", map_location=DEVICE))
        print("Loaded best_crnn_model.pth for prediction.")
    else:
        # ── Dataset ───────────────────────────────────────────────────────────
        df = pd.read_csv(TRAIN_LABEL_CSV)
        all_labels = list(zip(df["image"].astype(str), df["text"].astype(str)))
        random.shuffle(all_labels)

        val_n     = int(len(all_labels) * VAL_SPLIT)
        train_data = all_labels[val_n:]
        val_data   = all_labels[:val_n]

        kw = dict(num_workers=2, pin_memory=True, persistent_workers=True)
        train_loader = DataLoader(
            CaptchaDataset(TRAIN_IMG_DIR, train_data, augment=True),
            BATCH_SIZE, shuffle=True, collate_fn=collate_train, **kw
        )
        val_loader = DataLoader(
            CaptchaDataset(TRAIN_IMG_DIR, val_data, augment=False),
            BATCH_SIZE, shuffle=False, collate_fn=collate_train, **kw
        )

        print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

        # ── Optimiser & scheduler ─────────────────────────────────────────────
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        # Cosine annealing with warm restarts
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=20, T_mult=2, eta_min=1e-6
        )

        # ── Training ──────────────────────────────────────────────────────────
        best_cer, patience_cnt = float("inf"), 0
        print(f"{'Ep':>4}  {'TrLoss':>8}  {'TrCER':>7}  {'VlLoss':>8}  {'VlCER':>7}  {'LR':>9}")
        print("-" * 55)

        for ep in range(1, NUM_EPOCHS + 1):
            t0 = time.time()
            tr_loss, tr_cer = train_one_epoch(model, train_loader, optimizer)
            vl_loss, vl_cer = evaluate(model, val_loader)
            scheduler.step()

            cur_lr = optimizer.param_groups[0]["lr"]
            elapsed = time.time() - t0
            print(f"{ep:>4}  {tr_loss:>8.4f}  {tr_cer:>7.4f}  "
                  f"{vl_loss:>8.4f}  {vl_cer:>7.4f}  {cur_lr:>9.2e}  [{elapsed:.0f}s]")

            if vl_cer < best_cer:
                best_cer = vl_cer
                patience_cnt = 0
                torch.save(model.state_dict(), "best_crnn_model.pth")
                print(f"      ✓ Best model saved  (Val CER={best_cer:.4f})")
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE:
                    print(f"Early stop at epoch {ep}  (no improvement for {PATIENCE} epochs).")
                    break

        print(f"\nBest Val CER: {best_cer:.4f}")
        model.load_state_dict(torch.load("best_crnn_model.pth", map_location=DEVICE))

    # ── Inference ─────────────────────────────────────────────────────────────
    test_ds     = CaptchaDataset(TEST_IMG_DIR, labels=None, augment=False)
    test_loader = DataLoader(
        test_ds, BATCH_SIZE, shuffle=False,
        collate_fn=collate_test, num_workers=2
    )

    model.eval()
    results = []
    for imgs, fnames in test_loader:
        preds = predict_with_tta(model, imgs, n_aug=4)
        results.extend(zip(fnames, preds))

    out_csv = f"submission_{YOUR_NAME}_{YOUR_ENROLL}.csv"
    pd.DataFrame(results, columns=["image", "prediction"]).to_csv(out_csv, index=False)
    print(f"\nSubmission saved: {out_csv}  ({len(results)} rows)")
    return out_csv


out_csv = main(predict_only=False)

Train: 18000  |  Val: 2000
  Ep    TrLoss    TrCER    VlLoss    VlCER         LR
-------------------------------------------------------
   1    2.1842   0.6255    0.2185   0.0705   2.98e-04  [156s]
      ✓ Best model saved  (Val CER=0.0705)
   2    0.1022   0.0257    0.0284   0.0070   2.93e-04  [158s]
      ✓ Best model saved  (Val CER=0.0070)
   3    0.0524   0.0138    0.0062   0.0015   2.84e-04  [159s]
      ✓ Best model saved  (Val CER=0.0015)
   4    0.0385   0.0103    0.0088   0.0024   2.71e-04  [159s]
   5    0.0318   0.0085    0.0037   0.0008   2.56e-04  [158s]
      ✓ Best model saved  (Val CER=0.0008)
   6    0.0267   0.0077    0.0014   0.0003   2.38e-04  [159s]
      ✓ Best model saved  (Val CER=0.0003)
   7    0.0215   0.0059    0.0019   0.0004   2.18e-04  [159s]
   8    0.0211   0.0060    0.0009   0.0003   1.97e-04  [159s]
   9    0.0187   0.0054    0.0004   0.0000   1.74e-04  [158s]
      ✓ Best model saved  (Val CER=0.0000)
  10    0.0148   0.0041    0.0004   0.0001   1.

In [20]:
# ── 8. Quick sanity check on submission file ──────────────────────────────────
sub = pd.read_csv(out_csv)
print(f"Submission rows : {len(sub)}")
print(f"Sample predictions:")
print(sub.head(10).to_string(index=False))
print(f"\nPrediction length distribution:")
print(sub["prediction"].str.len().value_counts().sort_index())

Submission rows : 5000
Sample predictions:
        image prediction
   test-0.png     QVTQ8A
   test-1.png     7PSW9D
  test-10.png     7DUP98
 test-100.png     75Z4WT
test-1000.png     QAKZ7V
test-1001.png     R6MERY
test-1002.png     CHXX67
test-1003.png     9NV2WP
test-1004.png     F56TDZ
test-1005.png         TR

Prediction length distribution:
prediction
1       2
2      10
3      64
4     156
5     457
6    4311
Name: count, dtype: int64
